In [2]:
### QUESTIONS TABLE CREATION CODE ###

from pyspark.sql.functions import (
    col,
    regexp_replace,
    regexp_extract,
    trim,
    substring,
    input_file_name
)

# ============================================================
# UL Lakehouse Metadata
# ============================================================

ul_meta_df = spark.read.csv(
    "Files/Meta Data",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

ul_meta_df = ul_meta_df.withColumn(
    "input_file_name",
    regexp_extract(input_file_name(), r"([^/]+$)", 1)
)

ul_meta_df = ul_meta_df.withColumn(
    "Category",
    substring(col("input_file_name"), 5, 4)
)

# ============================================================
# Historical Metadata
# ============================================================

hist_meta_df = spark.read.csv(
    "Files/META DATA_202301_202630",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

hist_meta_df = hist_meta_df.withColumn(
    "input_file_name",
    regexp_extract(input_file_name(), r"([^/]+$)", 1)
)

hist_meta_df = hist_meta_df.withColumn(
    "Category",
    substring(col("input_file_name"), 5, 4)
)

# ============================================================
# July Metadata
# ============================================================

jul_meta_df = spark.read.csv(
    "Files/META DATA_202627_202630",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

jul_meta_df = jul_meta_df.withColumn(
    "input_file_name",
    regexp_extract(input_file_name(), r"([^/]+$)", 1)
)

jul_meta_df = jul_meta_df.withColumn(
    "Category",
    substring(col("input_file_name"), 5, 4)
)

# ============================================================
# Combine all Metadata
# ============================================================

df = (
    ul_meta_df
        .unionByName(hist_meta_df)
        .unionByName(jul_meta_df)
)

# QC
print("UL Metadata:", ul_meta_df.count())
print("Historical Metadata:", hist_meta_df.count())
print("July Metadata:", jul_meta_df.count())
print("Combined Metadata:", df.count())

# ============================================================
# CLEANING
# ============================================================

df = df.withColumn(
    "Top Level Field Name",
    regexp_replace("Top Level Field Name", "[^\\x00-\\x7F]+", "")
)

# Keep required variables only
df = df.filter(
    col("Top Level Field Name").isin([
        'BE6Total',
        'BE7Total',
        'BP1',
        'BP6',
        'Category',
        'COM19Flip',
        'Country',
        'Ethnicity',
        'MRKAgeBands',
        'Pack1Core',
        'Q7aYes',
        'QSex',
        'SOCMED1_CORE',
        'SOCMED2_CORE',
        'upos1',
        'upos2',
        'UsageTab',
        'UPOS2_HeardOf',
        'BHCAsked'
    ])
)

df = df.withColumn(
    "Response Label",
    regexp_replace("Response Label", "[^\\x00-\\x7F]+", "")
)

df = df.withColumn(
    "Response Label",
    regexp_replace("Response Label", '"', "")
)

df = df.withColumn(
    "Response Label",
    trim("Response Label")
)

df = df.withColumn(
    "First Level Iterator Label",
    regexp_replace(
        "First Level Iterator Label",
        "[^\\x00-\\x7F]+",
        ""
    )
)

df = df.withColumn(
    "Second Level Iterator Label",
    regexp_replace(
        "Second Level Iterator Label",
        "[^\\x00-\\x7F]+",
        ""
    )
)

# Keep only selected columns
df = df.select(
    'Category',
    'Top Level Field Name',
    'First Level Iterator Name',
    'First Level Iterator Label',
    'First Level Field Name',
    'Response Name',
    'Response Label',
    'Second Level Iterator Label',
    'Second Level Iterator Name',
    'Second Level Field Name'
)

df = df.dropDuplicates()

df = df.replace(
    {"Tresemm": "Tresemme"},
    subset=[
        "First Level Iterator Label",
        "Second Level Iterator Label",
        "Response Label"
    ]
)

print("Final Metadata Count:", df.count())

StatementMeta(, eb3b6d00-1141-4670-893f-7749a310afdc, 4, Finished, Available, Finished, False)

UL Metadata: 756102
Historical Metadata: 5932
July Metadata: 270582
Combined Metadata: 1032616
Final Metadata Count: 49823


In [2]:
df = spark.sql("SELECT * FROM UL_Synthetic.dbo.basetable_1 LIMIT 1000")
display(df)

StatementMeta(, d905ae46-841e-43be-ae1b-c7d112de21bf, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 90f0f5af-eb69-4e12-aac3-38a1a5321cfc)

In [3]:
# distinct Top Level Field Names
display(df.select('Top Level Field Name').distinct())


StatementMeta(, eb3b6d00-1141-4670-893f-7749a310afdc, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a6e19fc9-3168-4a27-aff8-444679e825d3)

In [4]:
df.coalesce(1).write.mode("overwrite").option("header", True).csv("Files/Synthetic_Metadata_processed")


StatementMeta(, eb3b6d00-1141-4670-893f-7749a310afdc, 6, Finished, Available, Finished, False)

In [5]:
# removing category column
df = df.drop('Category')

df = df.dropDuplicates()

df.count()

StatementMeta(, eb3b6d00-1141-4670-893f-7749a310afdc, 7, Finished, Available, Finished, False)

30066

In [6]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import row_number

# creating Questions_id

# Define a window specification
window_spec = Window.orderBy(
    col("Top Level Field Name"),
    col("First Level Iterator Name"),
    col("Second Level Iterator Name"),
    col("Response Name")
)

# Add Questions ID
df_1 = df.withColumn(
    "Questions ID",
    row_number().over(window_spec)
)

# display(df_1)

# Creating Brands Column

df_2 = df_1.withColumn(
    "Brands",
    F.when(
        F.col("Top Level Field Name").isin(
            ['BE6Total','BE7Total','BP6','COM19Flip','UPOS2_HeardOf']
        ),
        F.col("First Level Iterator Label")
    )
    .when(
        F.col("Top Level Field Name").isin(
            [
                'UsageTab',
                'Q7aYes',
                'upos2',
                'upos1',
                'Pack1Core',
                'BP1_Needs_T2B',
                'BP6_Worth_Price_TB',
                'BE6_T2B',
                'BE7Total_T2B',
                'BP1_FollowVsTrends_T2B',
                'BP1_Perceived_Price_T2B',
                'BP1_SameVsDifferent_T2B',
                'BHCAsked'
            ]
        ),
        F.col("Response Label")
    )
    .when(
        F.col("Top Level Field Name") == 'BP1',
        F.col("Second Level Iterator Label")
    )
    .otherwise("")
)

# display(df_2)

# Creating Statements Column
questions_df = df_2.withColumn(
    "Statements",
    F.when(
        F.col("Top Level Field Name").isin(
            ['BE6Total','BP6','COM19Flip','BE7Total','UPOS2_HeardOf']
        ),
        F.col("Response Label")
    )
    .when(
        F.col("Top Level Field Name").isin(['BP1']),
        F.concat(
            F.col("First Level Iterator Name"),
            F.col("Response Label")
        )
    )
    .when(
        F.col("Top Level Field Name").isin(
            ['UsageTab','upos1','upos2','Pack1Core']
        ),
        F.col("First Level Iterator Label")
    )
    .otherwise("")
)

display(questions_df)

StatementMeta(, eb3b6d00-1141-4670-893f-7749a310afdc, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e95ee695-78e0-4388-9ff7-bd8fb5e4f075)

In [7]:
questions_df.count()

### END QUESTIONS TABLE CREATION CODE ###


StatementMeta(, eb3b6d00-1141-4670-893f-7749a310afdc, 9, Finished, Available, Finished, False)

30066

In [ ]:
### CASEDATA TABLE CREATION CODE ### -- ABDULRAHMAN FOR CHECKING

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    regexp_extract,
    regexp_replace,
    input_file_name,
    trim,
    upper, substring
)

ul_case_df = spark.read.csv(
    "Files/Case Data",
    sep=",", inferSchema=True, header=True, encoding="UTF-16"
)

ul_case_df = ul_case_df.withColumn("input_file_name", input_file_name())

hist_case_df = spark.read.csv(
    "Files/CASE DATA_202301_202630",
    sep=",", inferSchema=True, header=True, encoding="UTF-16"
)

hist_case_df = hist_case_df.withColumn("input_file_name", input_file_name())

jul_case_df = spark.read.csv(
    "Files/CASE DATA_202627_202630",
    sep=",", inferSchema=True, header=True, encoding="UTF-16"
)

jul_case_df = jul_case_df.withColumn("input_file_name", input_file_name())

casedata_df = ul_case_df.union(hist_case_df).union(jul_case_df)

casedata_df = casedata_df.withColumn(
    "input_file_name", regexp_extract("input_file_name", r"([^/]+\.CSV)", 1)
)
casedata_df = casedata_df.withColumn("Category", substring("input_file_name", 5, 4))


StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 3, Finished, Available, Finished, False)

In [ ]:
### -- ABDULRAHMAN FOR CHECKING

casedata_df.select("Category","`Respondent.Serial`").distinct().count()

StatementMeta(, af93275a-66a8-480c-a043-72f0b241908f, 7, Finished, Available, Finished, False)

107997

In [ ]:
### -- ABDULRAHMAN FOR CHECKING

from pyspark.sql.functions import (
    concat_ws
)

casedata_df = casedata_df.filter(
    col('Top Level Field Name').isin([

        # MAIN METRICS
        'BE6Total','BE7Total','BP1',
        'COM19Flip','Q7aYes','upos2','UsageTab','UPOS2_HeardOf',

        # SYNTHETIC WEIGHTS
        'Synthetic','WEIGHT','Weight_human','boost_factor','human_weights_squared',

        # DEMOGRAPHICS
        'Country','Category','QSex','MRKAgeBands','Age',

        # ETHNICITY
        'Ethnicity',

        # DIGITAL PLATFORMS
        'SOCMED1_CORE',   # Digital Platform Regularly
        'SOCMED2_CORE',   # Digital Platform Daily

        # CALENDAR 
        'Rounds'
        
    ])
)

# Remove literal 'null'
casedata_df = casedata_df.filter(col("Response Name") != 'null')

# 1) Select the required columns (keep the backticked field as-is)
casedata_df = casedata_df.select([
    "`Respondent.Serial`",
    "Week number",
    "Top Level Field Name",
    "First Level Iterator Name",
    "First Level Iterator Label",
    "First Level Field Name",
    "Response Name",
    "Second Level Iterator Label",
    "Second Level Iterator Name",
    "Second Level Field Name",
    "Category"
])

# 2) Add Respondent Category ID = <Category-from-filename> - <Respondent.Serial>
#    Category is still derived as substring(file_name, 5, 4) per your earlier logic.
casedata_df = casedata_df.withColumn(
    "Respondent Category ID",
    concat_ws(
        " - ",
        "Category",
        col("`Respondent.Serial`")
    )
)

# 3) Preview
display(casedata_df)

StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 47de2c9e-8eb1-4cd9-ba7e-0189dbe624fd)

In [ ]:
### -- ABDULRAHMAN FOR CHECKING

casedata_df.select("Respondent Category ID").distinct().count()

StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 5, Finished, Available, Finished, False)

107983

In [ ]:
### -- ABDULRAHMAN FOR CHECKING

casedata_df.select("Category","`Respondent.Serial`").distinct().count()

StatementMeta(, af93275a-66a8-480c-a043-72f0b241908f, 18, Finished, Available, Finished, False)

107983

In [ ]:
### -- ABDULRAHMAN FOR CHECKING

# 4) CLEAN TEXT COLUMNS
casedata_df = casedata_df.withColumn(
    "Response Name", regexp_replace("Response Name", r"[^\x00-\x7F]", "")
)
casedata_df = casedata_df.withColumn(
    "Top Level Field Name", regexp_replace("Top Level Field Name", r"[^\x00-\x7F]", "")
)
casedata_df = casedata_df.withColumn(
    "First Level Iterator Label", regexp_replace("First Level Iterator Label", r"[^\x00-\x7F]", "")
)
casedata_df = casedata_df.withColumn(
    "Second Level Iterator Label", regexp_replace("Second Level Iterator Label", r"[^\x00-\x7F]", "")
)

# Remove literal 'null'
casedata_df = casedata_df.filter(col("Response Name") != 'null')

# 5) Standardize brand names
casedata_df = casedata_df.replace(
    {"Tresemm": "Tresemme"},
    subset=["First Level Iterator Label", "Second Level Iterator Label"]
)

casedata_df = casedata_df.dropDuplicates()

StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 6, Finished, Available, Finished, False)

In [ ]:
### -- ABDULRAHMAN FOR CHECKING

from pyspark.sql.functions import collect_list,concat_ws

# Seperating and pivoting weights and others

# -----------------------------
# 1) Keep only variables to be pivoted
# -----------------------------
weights_df = (
    casedata_df
      .filter(col("Top Level Field Name").isin(['Synthetic','WEIGHT','Weight_human','human_weights_squared']))
)

# 2) Take the columns you need
weights_df = weights_df.select(
    col("Respondent Category ID"),
    col("Week number"),
    col("Top Level Field Name"),
    col("Response Name")
)


weights_df.select("Respondent Category ID").distinct().count()


StatementMeta(, af93275a-66a8-480c-a043-72f0b241908f, 23, Finished, Available, Finished, False)

107983

In [ ]:
### -- ABDULRAHMAN FOR CHECKING

weights_df.select("Respondent Category ID", "Week number").distinct().count()

StatementMeta(, af93275a-66a8-480c-a043-72f0b241908f, 25, Finished, Available, Finished, False)

112118

In [ ]:
### -- ABDULRAHMAN FOR CHECKING

# 3) Pivot on Top Level Field Name
weights_df = (
    weights_df
      .groupBy(col("Respondent Category ID"),col("Week number"))
      .pivot("Top Level Field Name")
      .agg(concat_ws(", ", collect_list(col("Response Name"))))
)

weights_df.coalesce(1).write.mode("overwrite").option("header", True).csv("Files/weights_df")
#display(weights_df)

#weights_df.count()

StatementMeta(, af93275a-66a8-480c-a043-72f0b241908f, 26, Finished, Available, Finished, False)

In [ ]:
### -- ABDULRAHMAN FOR CHECKING 
# boost factor

bf_df = (
    casedata_df
      .filter(col("Top Level Field Name").isin(['Rounds', "boost_factor"]))
)

# Select required columns
bf_df = bf_df.select(
    col("Category"),
    col("Week number"),
    col("Top Level Field Name"),
    col("Response Name")
)

bf_df = bf_df.dropDuplicates()


#bf_df.select("Respondent Category ID").distinct().count()


StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 15, Finished, Available, Finished, False)

In [ ]:
### -- ABDULRAHMAN FOR CHECKING 
from pyspark.sql.functions import collect_list,concat_ws
# Pivot -> column 'boost_factor'
bf_df = (
    bf_df
      .groupBy(col("Category"),col("Week number"))
      .pivot("Top Level Field Name")
      .agg(concat_ws(", ", collect_list(col("Response Name"))))
)

display(bf_df.select("Category","boost_factor").distinct())

StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 302709f4-e0e8-4df8-ac80-737e73126335)

In [ ]:
### -- ABDULRAHMAN FOR CHECKING 
# base table

base_df = casedata_df.filter(col('Top Level Field Name').isin(['Country','Category','QSex','MRKAgeBands','Age']))

base_df = base_df.select(["Week number","`Top Level Field Name`","`Response Name`","`Respondent Category ID`"])

base_df.select("Respondent Category ID").distinct().count()

StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 17, Finished, Available, Finished, False)

103533

In [ ]:
### -- ABDULRAHMAN FOR CHECKING 

casedata_df.select("Respondent Category ID").distinct().count()

StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 18, Finished, Available, Finished, False)

107983

In [ ]:
### -- ABDULRAHMAN FOR CHECKING 

# Pivot
base_df = base_df.groupBy(["`Respondent Category ID`","Week number",]).pivot("`Top Level Field Name`").agg(concat_ws(", ", collect_list("`Response Name`")))

base_df.select("Respondent Category ID").distinct().count()

StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 19, Finished, Available, Finished, False)

103533

In [ ]:
### -- ABDULRAHMAN FOR CHECKING 

base_df.count()

StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 20, Finished, Available, Finished, False)

107668

In [ ]:
### -- ABDULRAHMAN FOR CHECKING 

base_df.coalesce(1).write.mode("overwrite").option("header", True).csv("Files/base_df")


StatementMeta(, 983697db-48c2-4424-8e27-5ce1404b92ed, 21, Finished, Available, Finished, False)

## Case Data Table Creation

In [8]:
### CASEDATA TABLE CREATION CODE ###

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    regexp_extract,
    regexp_replace,
    input_file_name,
    trim,
    upper
)

# ============================================================
# CONFIGURATION
# ============================================================

valid_categories = [
    "FBSN",
    "FMON",
    "FMSN",
    "FSIN",
    "FSMB",
    "HBSF",
    "HBWF",
    "HBWP",
    "HDOF",
    "HDOM",
    "HFCN",
    "HHBL",
    "HSKM",
    "HSYF"
]

category_regex = (
    r"(?i)(FBSN|FMON|FMSN|FSIN|FSMB|HBSF|HBWF|HBWP|"
    r"HDOF|HDOM|HFCN|HHBL|HSKM|HSYF)"
)

required_variables = [
    "BE6Total",
    "BE7Total",
    "BP1",
    "BP6", #
    "WEIGHT",
    "COM19Flip",
    "Pack1Core", #
    "Q7aYes",
    "upos1", #
    "upos2",
    "UsageTab",
    "UPOS2_HeardOf",
    "BHCAsked", #
    "Country",
    "Category",
    "QSex",
    "MRKAgeBands",
    "Age",
    "Ethnicity",
    "SOCMED1_CORE",
    "SOCMED2_CORE",
    "Synthetic",
    "Weight_human",
    "human_weights_squared",
    "boost_factor",
    "Rounds"
]

# ============================================================
# HELPER FUNCTION
# READ EACH SOURCE AND CAPTURE CATEGORY BEFORE UNION
# ============================================================

def read_casedata_with_category(source_path, source_group):

    source_df = spark.read.csv(
        source_path,
        sep=",",
        inferSchema=True,
        header=True,
        encoding="UTF-16"
    )

    source_df = source_df.withColumn(
        "Source_Path",
        input_file_name()
    )

    source_df = source_df.withColumn(
        "Source_File_Name",
        regexp_extract(
            col("Source_Path"),
            r"([^/\\]+$)",
            1
        )
    )

    source_df = source_df.withColumn(
        "Category",
        upper(
            regexp_extract(
                col("Source_File_Name"),
                category_regex,
                1
            )
        )
    )

    source_df = source_df.withColumn(
        "Source_Group",
        F.lit(source_group)
    )

    return source_df

# ============================================================
# 1. READ ALL THREE CASEDATA SOURCES
# ============================================================

ul_case_df = read_casedata_with_category(
    "Files/Case Data",
    "UL"
)

hist_case_df = read_casedata_with_category(
    "Files/CASE DATA_202301_202630",
    "Historical"
)

jul_case_df = read_casedata_with_category(
    "Files/CASE DATA_202627_202630",
    "July"
)

# ============================================================
# 2. VALIDATE SOURCE FILE CATEGORY EXTRACTION
# ============================================================

source_file_qc_df = (
    ul_case_df
        .select(
            "Source_Group",
            "Source_File_Name",
            "Category"
        )
        .unionByName(
            hist_case_df.select(
                "Source_Group",
                "Source_File_Name",
                "Category"
            )
        )
        .unionByName(
            jul_case_df.select(
                "Source_Group",
                "Source_File_Name",
                "Category"
            )
        )
        .distinct()
)

invalid_source_files_df = (
    source_file_qc_df
        .filter(
            col("Category").isNull()
            | (trim(col("Category")) == "")
            | (~col("Category").isin(valid_categories))
        )
)

invalid_source_file_count = invalid_source_files_df.count()

print(
    "Source files with missing or invalid Category:",
    invalid_source_file_count
)

if invalid_source_file_count > 0:
    display(
        invalid_source_files_df.orderBy(
            "Source_Group",
            "Source_File_Name"
        )
    )

if invalid_source_file_count > 0:
    raise ValueError(
        "Category could not be extracted from one or more CaseData "
        "filenames. Review the displayed source files before continuing."
    )

# ============================================================
# 3. COMBINE ALL CASEDATA SOURCES
# ============================================================

casedata_df = (
    ul_case_df
        .unionByName(
            hist_case_df,
            allowMissingColumns=True
        )
        .unionByName(
            jul_case_df,
            allowMissingColumns=True
        )
)

# ============================================================
# 4. CLEAN REQUIRED TEXT FIELDS
# ============================================================

casedata_df = casedata_df.withColumn(
    "Top Level Field Name",
    trim(
        regexp_replace(
            col("Top Level Field Name"),
            r"[^\x00-\x7F]+",
            ""
        )
    )
)

casedata_df = casedata_df.withColumn(
    "Response Name",
    trim(
        regexp_replace(
            col("Response Name"),
            r"[^\x00-\x7F]+",
            ""
        )
    )
)

casedata_df = casedata_df.withColumn(
    "First Level Iterator Label",
    regexp_replace(
        col("First Level Iterator Label"),
        r"[^\x00-\x7F]+",
        ""
    )
)

casedata_df = casedata_df.withColumn(
    "Second Level Iterator Label",
    regexp_replace(
        col("Second Level Iterator Label"),
        r"[^\x00-\x7F]+",
        ""
    )
)

# ============================================================
# 5. KEEP REQUIRED VARIABLES
# ============================================================

casedata_df = casedata_df.filter(
    col("Top Level Field Name").isin(required_variables)
)

# ============================================================
# 6. REMOVE NULL / BLANK RESPONSE RECORDS
# ============================================================

casedata_df = casedata_df.filter(
    col("Response Name").isNotNull()
    & (trim(col("Response Name")) != "")
    & (F.lower(trim(col("Response Name"))) != "null")
)

# ============================================================
# 7. STANDARDISE BRAND NAMES
# ============================================================

casedata_df = casedata_df.replace(
    {
        "Tresemm": "Tresemme"
    },
    subset=[
        "First Level Iterator Label",
        "Second Level Iterator Label"
    ]
)

# ============================================================
# 8. REMOVE SOURCE COLUMNS BEFORE DEDUPLICATION
#
# This allows genuinely overlapping rows from the three source
# folders to be removed even when their source paths differ.
# Category is retained as part of the row identity.
# ============================================================

casedata_df = casedata_df.drop(
    "Source_Path",
    "Source_File_Name",
    "Source_Group"
)

casedata_df = casedata_df.dropDuplicates()

# ============================================================
# 9. FINAL CATEGORY QC
# ============================================================

invalid_category_count = (
    casedata_df
        .filter(
            col("Category").isNull()
            | (trim(col("Category")) == "")
            | (~col("Category").isin(valid_categories))
        )
        .count()
)

print(
    "CaseData records with missing or invalid Category:",
    invalid_category_count
)

if invalid_category_count > 0:
    raise ValueError(
        "CaseData contains missing or invalid Category values. "
        "Respondent Category ID must not be created."
    )

display(
    casedata_df
        .groupBy("Category")
        .agg(
            F.countDistinct(
                col("`Respondent.Serial`")
            ).alias("Distinct_Respondents"),
            F.countDistinct(
                col("Week number")
            ).alias("Distinct_Weeks")
        )
        .orderBy("Category")
)

display(
    casedata_df
        .select("Top Level Field Name")
        .distinct()
        .orderBy("Top Level Field Name")
)

StatementMeta(, eb3b6d00-1141-4670-893f-7749a310afdc, 10, Submitted, Running, Running, True)

In [ ]:
check_df = casedata_df.select(
    "Category",
    "Top Level Field Name",
    "First Level Iterator Name",
    "First Level Iterator Label",
    "First Level Field Name",
    "Response Name",
    "Second Level Iterator Label",
    "Second Level Iterator Name",
    "Second Level Field Name"
).distinct()

check_df.coalesce(1).write.mode("overwrite").option("header", True).csv("Files/Synthetic_Casedata_processed")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, concat_ws, trim, upper

# Select required columns
casedata_df = casedata_df.select(
    col("`Respondent.Serial`"),
    col("Week number"),
    col("Top Level Field Name"),
    col("First Level Iterator Name"),
    col("First Level Iterator Label"),
    col("First Level Field Name"),
    col("Response Name"),
    col("Second Level Iterator Label"),
    col("Second Level Iterator Name"),
    col("Second Level Field Name"),
    col("Category")
)

# Create Respondent Category ID
# Required format: HDOM-223020074
casedata_df = casedata_df.withColumn(
    "Respondent Category ID",
    concat_ws(
        "-",
        upper(trim(col("Category"))),
        col("`Respondent.Serial`").cast("string")
    )
)

# ============================================================
# QC 1 : INVALID IDS
# ============================================================

invalid_respondent_id_count = (
    casedata_df
        .filter(
            col("Category").isNull()
            | (trim(col("Category")) == "")
            | col("Respondent Category ID").isNull()
            | col("Respondent Category ID").rlike(r"^-")
        )
        .count()
)

print(
    "Invalid Respondent Category ID records:",
    invalid_respondent_id_count
)

# ============================================================
# QC 2 : ONE CATEGORY PER ID
# ============================================================

category_conflict_df = (
    casedata_df
        .groupBy("Respondent Category ID")
        .agg(
            F.countDistinct("Category").alias("Category_Count"),
            F.collect_set("Category").alias("Categories")
        )
        .filter(col("Category_Count") > 1)
)

category_conflict_count = category_conflict_df.count()

print(
    "Respondent IDs mapped to multiple Categories:",
    category_conflict_count
)

if category_conflict_count > 0:
    display(category_conflict_df)

# ============================================================
# QC 3 : SAMPLE OUTPUT
# ============================================================

display(
    casedata_df
        .select(
            col("Category"),
            col("`Respondent.Serial`"),
            col("Respondent Category ID")
        )
        .distinct()
        .orderBy(
            col("Category"),
            col("`Respondent.Serial`")
        )
        .limit(100)
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Creation of new unique IDs

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
casedata_df.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# ============================================================
# WEIGHTS TABLE CREATION
# ============================================================

weight_variables = [
    "Synthetic",
    "WEIGHT",
    "Weight_human",
    "human_weights_squared"
]

# ============================================================
# 1. KEEP ONLY WEIGHT VARIABLES
# ============================================================

weights_df = (
    casedata_df
        .filter(
            col("Top Level Field Name").isin(weight_variables)
        )
        .select(
            col("Respondent Category ID"),
            col("Week number"),
            col("Top Level Field Name"),
            col("Response Name")
        )
        .dropDuplicates()
)

# ============================================================
# 2. QC - CHECK MULTIPLE VALUES PER KEY
# ============================================================

weight_conflicts_df = (
    weights_df
        .groupBy(
            "Respondent Category ID",
            "Week number",
            "Top Level Field Name"
        )
        .agg(
            F.countDistinct("Response Name").alias(
                "Distinct_Value_Count"
            ),
            F.collect_set("Response Name").alias(
                "Values"
            )
        )
        .filter(
            col("Distinct_Value_Count") > 1
        )
)

weight_conflict_count = weight_conflicts_df.count()

print(
    "Weight conflicts found:",
    weight_conflict_count
)

if weight_conflict_count > 0:
    display(weight_conflicts_df)

# ============================================================
# 3. PIVOT USING FIRST()
# ============================================================

weights_df = (
    weights_df
        .groupBy(
            "Respondent Category ID",
            "Week number"
        )
        .pivot(
            "Top Level Field Name",
            weight_variables
        )
        .agg(
            F.first(
                "Response Name",
                ignorenulls=True
            )
        )
)

# ============================================================
# 4. CAST NUMERIC COLUMNS
# ============================================================

weights_df = (
    weights_df
        .withColumn(
            "WEIGHT",
            col("WEIGHT").cast("double")
        )
        .withColumn(
            "Weight_human",
            col("Weight_human").cast("double")
        )
        .withColumn(
            "human_weights_squared",
            col("human_weights_squared").cast("double")
        )
)

# ============================================================
# 5. QC - NULL COUNTS
# ============================================================

print("\n============ NULL COUNTS ============")

for c in weight_variables:

    if c in weights_df.columns:

        null_count = (
            weights_df
                .filter(col(c).isNull())
                .count()
        )

        print(
            f"{c} null records: {null_count}"
        )

# ============================================================
# 6. QC - NO COMMA SEPARATED VALUES
# ============================================================

print("\n============ COMMA SEPARATED VALUE CHECK ============")

for c in [
    "WEIGHT",
    "Synthetic",
    "Weight_human",
    "human_weights_squared"
]:

    bad_count = (
        weights_df
            .filter(
                col(c)
                    .cast("string")
                    .contains(",")
            )
            .count()
    )

    print(
        f"{c} comma-separated rows: {bad_count}"
    )

# ============================================================
# 7. QC - RESPONDENT KEY UNIQUENESS
# ============================================================

duplicate_keys_count = (
    weights_df
        .groupBy(
            "Respondent Category ID",
            "Week number"
        )
        .count()
        .filter(col("count") > 1)
        .count()
)

print(
    "\nDuplicate respondent-week keys:",
    duplicate_keys_count
)

# ============================================================
# 8. FINAL OUTPUT
# ============================================================

print("\n============ FINAL WEIGHTS SCHEMA ============")
print(weights_df.columns)

display(weights_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
weights_df.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# joining both weights to casedata table
fact_df = (
    casedata_df
        .join(weights_df, on=["Respondent Category ID", "Week number"], how="left")
    #    .join(poc_df,     on=["Respondent Category ID"], how="left")   # <-- NEW
)

display(fact_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# ============================================================
# BOOST FACTOR / ROUNDS TABLE
# ============================================================

bf_variables = [
    "Rounds",
    "boost_factor"
]

# ============================================================
# 1. KEEP REQUIRED VARIABLES
# ============================================================

bf_df = (
    casedata_df
        .filter(
            col("Top Level Field Name").isin(
                bf_variables
            )
        )
        .select(
            col("Category"),
            col("Week number"),
            col("Top Level Field Name"),
            col("Response Name")
        )
        .dropDuplicates()
)

# ============================================================
# 2. QC - CHECK MULTIPLE VALUES PER CATEGORY/WEEK
# ============================================================

bf_conflicts_df = (
    bf_df
        .groupBy(
            "Category",
            "Week number",
            "Top Level Field Name"
        )
        .agg(
            F.countDistinct("Response Name").alias(
                "Distinct_Value_Count"
            ),
            F.collect_set("Response Name").alias(
                "Values"
            )
        )
        .filter(
            col("Distinct_Value_Count") > 1
        )
)

bf_conflict_count = bf_conflicts_df.count()

print(
    "Rounds / Boost Factor Conflicts:",
    bf_conflict_count
)

if bf_conflict_count > 0:
    display(bf_conflicts_df)

# ============================================================
# 3. PIVOT
# ============================================================

bf_df = (
    bf_df
        .groupBy(
            "Category",
            "Week number"
        )
        .pivot(
            "Top Level Field Name",
            bf_variables
        )
        .agg(
            F.first(
                "Response Name",
                ignorenulls=True
            )
        )
)

# ============================================================
# 4. OPTIONAL CAST
# ============================================================

if "boost_factor" in bf_df.columns:

    bf_df = bf_df.withColumn(
        "boost_factor",
        col("boost_factor").cast("double")
    )

# ============================================================
# 5. QC - NULLS
# ============================================================

print("\n============ NULL COUNTS ============")

for c in bf_variables:

    if c in bf_df.columns:

        null_count = (
            bf_df
                .filter(col(c).isNull())
                .count()
        )

        print(
            f"{c} null records: {null_count}"
        )

# ============================================================
# 6. QC - DUPLICATE CATEGORY/WEEK KEYS
# ============================================================

duplicate_keys_count = (
    bf_df
        .groupBy(
            "Category",
            "Week number"
        )
        .count()
        .filter(col("count") > 1)
        .count()
)

print(
    "Duplicate Category-Week Keys:",
    duplicate_keys_count
)

display(bf_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# ============================================================
# JOIN BOOST FACTOR / ROUNDS TO FACT TABLE
# ============================================================

fact_rows_before_join = fact_df.count()

fact_df = (
    fact_df
        .join(
            bf_df,
            on=[
                "Category",
                "Week number"
            ],
            how="left"
        )
)

fact_rows_after_join = fact_df.count()

# ============================================================
# QC 1 - ROW COUNT VALIDATION
# ============================================================

print(
    "Fact rows before bf_df join:",
    fact_rows_before_join
)

print(
    "Fact rows after bf_df join :",
    fact_rows_after_join
)

print(
    "Row difference :",
    fact_rows_after_join - fact_rows_before_join
)

# ============================================================
# QC 2 - CATEGORY.1 CHECK
# ============================================================

unexpected_category_columns = [
    c for c in fact_df.columns
    if c.startswith("Category.")
]

print(
    "Unexpected Category columns:",
    unexpected_category_columns
)

# ============================================================
# QC 3 - DUPLICATE FACT KEYS
# ============================================================

duplicate_fact_keys = (
    fact_df
        .groupBy(
            "Respondent Category ID",
            "Week number",
            "Top Level Field Name",
            "Response Name"
        )
        .count()
        .filter(col("count") > 1)
)

print(
    "Duplicate fact keys:",
    duplicate_fact_keys.count()
)

# ============================================================
# QC 4 - NULL BOOST FACTORS
# ============================================================

print(
    "Null boost_factor rows:",
    fact_df
        .filter(
            col("boost_factor").isNull()
        )
        .count()
)

display(fact_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# step to remove weight & demography variables from top level field name

fact_df = fact_df.filter(col('Top Level Field Name').isin([
    'BE6Total',
    'BE7Total',
    'BP1',
    'BP6',
    'COM19Flip',
    'Pack1Core',
    'Q7aYes',
    'upos1',
    'upos2',
    'UsageTab',
    'UPOS2_HeardOf',
    'BHCAsked',
    'SOCMED1_CORE',
    'SOCMED2_CORE'
]))

print("Fact rows after variable filter:", fact_df.count())

display(
    fact_df
        .groupBy("Top Level Field Name")
        .count()
        .orderBy("Top Level Field Name")
)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    concat_ws,
    coalesce,
    lit,
    upper
)

# ============================================================
# STEP 1 - CLEAN QUESTIONS TABLE
# ============================================================

questions_df = questions_df.dropDuplicates()

# ============================================================
# STEP 2 - BUILD QUESTIONS KEY
# ============================================================

questions_df_1 = (
    questions_df
        .withColumn(
            "key",
            upper(
                concat_ws(
                    "_",
                    col("Top Level Field Name"),
                    coalesce(col("First Level Iterator Name"), lit("NA")),
                    coalesce(col("First Level Iterator Label"), lit("NA")),
                    coalesce(col("First Level Field Name"), lit("NA")),
                    coalesce(col("Second Level Iterator Name"), lit("NA")),
                    coalesce(col("Second Level Iterator Label"), lit("NA")),
                    coalesce(col("Second Level Field Name"), lit("NA")),
                    coalesce(col("Response Name"), lit("NA"))
                )
            )
        )
)

# ============================================================
# QC 1 - QUESTIONS KEY UNIQUENESS
# ============================================================

duplicate_question_keys = (
    questions_df_1
        .groupBy("key")
        .count()
        .filter(col("count") > 1)
)

duplicate_question_key_count = duplicate_question_keys.count()

print(
    "Duplicate question keys:",
    duplicate_question_key_count
)

if duplicate_question_key_count > 0:
    display(duplicate_question_keys)

questions_df_1 = questions_df_1.select(
    col("key"),
    col("Questions ID")
)

# ============================================================
# STEP 3 - BUILD FACT KEY
# ============================================================

fact_df_1 = (
    fact_df
        .withColumn(
            "key",
            upper(
                concat_ws(
                    "_",
                    col("Top Level Field Name"),
                    coalesce(col("First Level Iterator Name"), lit("NA")),
                    coalesce(col("First Level Iterator Label"), lit("NA")),
                    coalesce(col("First Level Field Name"), lit("NA")),
                    coalesce(col("Second Level Iterator Name"), lit("NA")),
                    coalesce(col("Second Level Iterator Label"), lit("NA")),
                    coalesce(col("Second Level Field Name"), lit("NA")),
                    coalesce(col("Response Name"), lit("NA"))
                )
            )
        )
)

# ============================================================
# STEP 4 - JOIN QUESTIONS
# ============================================================

fact_rows_before_join = fact_df_1.count()

fact_df_1 = (
    fact_df_1
        .join(
            questions_df_1,
            on="key",
            how="left"
        )
)

fact_rows_after_join = fact_df_1.count()

print(
    "Fact rows before Questions join:",
    fact_rows_before_join
)

print(
    "Fact rows after Questions join:",
    fact_rows_after_join
)

print(
    "Row difference:",
    fact_rows_after_join - fact_rows_before_join
)

# ============================================================
# QC 2 - MISSING QUESTION IDS
# ============================================================

missing_question_ids = (
    fact_df_1
        .filter(
            col("Questions ID").isNull()
        )
        .count()
)

print(
    "Rows with missing Questions ID:",
    missing_question_ids
)

# ============================================================
# STEP 5 - FINAL SELECT
# ============================================================

fact_df_1 = fact_df_1.select(
    col("Respondent Category ID"),
    col("`Respondent.Serial`").alias("Respondent.Serial"),
    col("`Week number`").alias("WeekNumber"),
    col("Rounds"),
    col("`Top Level Field Name`").alias("TopLevelField"),
    col("`First Level Iterator Name`").alias("IteratorName"),
    col("`First Level Iterator Label`").alias("IteratorLabel"),
    col("`First Level Field Name`").alias("FieldName"),
    col("`Second Level Iterator Name`").alias("SecondLevelIteratorName"),
    col("`Second Level Iterator Label`").alias("SecondLevelIteratorLabel"),
    col("`Second Level Field Name`").alias("SecondLevelFieldName"),
    col("`Questions ID`").alias("QuestionsID"),
    col("`Response Name`").alias("ResponseName"),
    col("WEIGHT").cast("double").alias("WEIGHT"),
    col("Weight_human").cast("double").alias("Weight_human"),
    col("human_weights_squared").cast("double").alias("human_weights_squared"),
    col("Synthetic"),
    col("boost_factor").cast("double").alias("boost_factor")
)

# ============================================================
# STEP 6 - REMOVE HELPER KEY
# ============================================================

fact_df_1 = fact_df_1.drop("key")

# ============================================================
# QC 3 - FINAL DUPLICATE FACT KEYS
# ============================================================

duplicate_fact_keys = (
    fact_df_1
        .groupBy(
            "Respondent Category ID",
            "WeekNumber",
            "TopLevelField",
            "ResponseName"
        )
        .count()
        .filter(col("count") > 1)
)

print(
    "Final duplicate fact keys:",
    duplicate_fact_keys.count()
)

display(fact_df_1)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
display(questions_df_1.groupBy("key").count().filter("count > 1"))

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
fact_df_1.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
### WRITING QUESTIONS TABLE ###
# replace spaces in column names to remove invalid characters
questions_df = questions_df.toDF(*(c.replace(' ', '_') for c in questions_df.columns))

# Now, save the DataFrames
questions_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("QuestionTable_1")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
### WRITING FACT TABLE ###
# replace spaces in column names to remove invalid characters
fact_df_1 = fact_df_1.toDF(*(c.replace(' ', '_') for c in fact_df_1.columns))

# Now, save the DataFrame
fact_df_1.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactTable_1")


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# ============================================================
# BASE / DEMO TABLE
# ============================================================

demo_variables = [
    "Country",
    "Category",
    "QSex",
    "MRKAgeBands",
    "Age"
]

# ============================================================
# 1. KEEP ONLY DEMOGRAPHIC VARIABLES
# ============================================================

base_df = (
    casedata_df
        .filter(
            col("Top Level Field Name").isin(
                demo_variables
            )
        )
        .select(
            col("Respondent Category ID"),
            col("Week number"),
            col("Top Level Field Name"),
            col("Response Name")
        )
        .dropDuplicates()
)

# ============================================================
# 2. QC - CHECK MULTIPLE VALUES PER DEMOGRAPHIC
# ============================================================

demo_conflicts_df = (
    base_df
        .groupBy(
            "Respondent Category ID",
            "Week number",
            "Top Level Field Name"
        )
        .agg(
            F.countDistinct("Response Name").alias(
                "Distinct_Value_Count"
            ),
            F.collect_set("Response Name").alias(
                "Values"
            )
        )
        .filter(
            col("Distinct_Value_Count") > 1
        )
)

demo_conflict_count = demo_conflicts_df.count()

print(
    "Demographic conflicts found:",
    demo_conflict_count
)

if demo_conflict_count > 0:
    display(
        demo_conflicts_df.orderBy(
            "Respondent Category ID",
            "Week number",
            "Top Level Field Name"
        )
    )

# ============================================================
# 3. SAFE PIVOT USING FIRST()
# ============================================================

base_df = (
    base_df
        .groupBy(
            "Respondent Category ID",
            "Week number"
        )
        .pivot(
            "Top Level Field Name",
            demo_variables
        )
        .agg(
            F.first(
                "Response Name",
                ignorenulls=True
            )
        )
)

# ============================================================
# 4. QC - NULL COUNTS
# ============================================================

print("\n============ NULL COUNTS ============")

for c in demo_variables:

    if c in base_df.columns:

        null_count = (
            base_df
                .filter(col(c).isNull())
                .count()
        )

        print(
            f"{c} null records: {null_count}"
        )

# ============================================================
# 5. QC - CHECK FOR COMMA SEPARATED VALUES
# ============================================================

print("\n============ COMMA SEPARATED VALUE CHECK ============")

for c in demo_variables:

    if c in base_df.columns:

        bad_count = (
            base_df
                .filter(
                    col(c)
                        .cast("string")
                        .contains(",")
                )
                .count()
        )

        print(
            f"{c} comma-separated rows: {bad_count}"
        )

# ============================================================
# 6. QC - DUPLICATE RESPONDENT KEYS
# ============================================================

duplicate_keys_count = (
    base_df
        .groupBy(
            "Respondent Category ID",
            "Week number"
        )
        .count()
        .filter(col("count") > 1)
        .count()
)

print(
    "\nDuplicate respondent-week keys:",
    duplicate_keys_count
)

# ============================================================
# 7. QC - CATEGORY CONSISTENCY
# ============================================================

category_conflict_count = (
    base_df
        .groupBy("Respondent Category ID")
        .agg(
            F.countDistinct("Category").alias(
                "Category_Count"
            )
        )
        .filter(
            col("Category_Count") > 1
        )
        .count()
)

print(
    "Respondent IDs with multiple Categories:",
    category_conflict_count
)

# ============================================================
# 8. FINAL OUTPUT
# ============================================================

display(base_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
base_df.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
### WRITING BASE/DEMO TABLE ###
# replace spaces in column names to remove invalid characters

base_df = base_df.toDF(*(c.replace(' ', '_') for c in base_df.columns))

# Now, save the DataFrame

base_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("BaseTable_1")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Creating Ethnicity table from casedata

eth_df = casedata_df.filter(col("Top Level Field Name") == "Ethnicity")

eth_df = eth_df.select(
    col("Respondent Category ID"),
    col("Week number"),
    col("Response Name").alias("Ethnicity")
)

display(eth_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
#eth_df.select("Respondent Category ID").distinct().count()
display(eth_df.select("Ethnicity").distinct())

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
### WRITING ETHNICITY TABLE ###
# replace spaces in column names to remove invalid characters
eth_df = eth_df.toDF(*(c.replace(' ', '_') for c in eth_df.columns))
# write table
eth_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("EthnicityTable_1")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Creating Digital Platform Regularly table from casedata

dpr_df = casedata_df.filter(col("Top Level Field Name") == "SOCMED1_CORE")

dpr_df = dpr_df.select(
    col("Respondent Category ID"),
    col("Week number"),
    col("Top Level Field Name"),
    col("Response Name").alias("DPR Name")
)

display(dpr_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
dpr_df.select("Respondent Category ID").distinct().count()


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
### WRITING DIGITAL PLATFORM TAG REGULARLY TABLE ###
# replace spaces in column names to remove invalid characters
dpr_df = dpr_df.toDF(*(c.replace(' ', '_') for c in dpr_df.columns))
# write table
dpr_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DPTRegularlyTable_1")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Creating Digital Platform Daily table from casedata

dpd_df = casedata_df.filter(col("Top Level Field Name") == "SOCMED2_CORE")

dpd_df = dpd_df.select(
    col("Respondent Category ID"),
    col("Week number"),
    col("Top Level Field Name"),
    col("Response Name").alias("DPD Name")
)

display(dpd_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
dpd_df.select("Respondent Category ID").distinct().count()


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
### WRITING DIGITAL PLATFORM TAG DAILY TABLE ###
# replace spaces in column names to remove invalid characters
dpd_df = dpd_df.toDF(*(c.replace(' ', '_') for c in dpd_df.columns))
# write table
dpd_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DPTDailyTable_1")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
### CODE TO CHECK WEEK NUMBERS IN ALL FILES ###

from pyspark.sql.functions import input_file_name, regexp_extract

# UL Lakehouse CaseData
ul_temp_df = spark.read.csv(
    "Files/Case Data",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

# Historical CaseData
hist_temp_df = spark.read.csv(
    "Files/CASE DATA_202301_202630",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

# July CaseData
jul_temp_df = spark.read.csv(
    "Files/CASE DATA_202627_202630",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

# Combine all CaseData
temp_df = (
    ul_temp_df
        .unionByName(hist_temp_df)
        .unionByName(jul_temp_df)
)

# QC
print("UL CaseData:", ul_temp_df.count())
print("Historical CaseData:", hist_temp_df.count())
print("July CaseData:", jul_temp_df.count())
print("Combined CaseData:", temp_df.count())

temp_df = temp_df.withColumn(
    "input_file_name",
    input_file_name()
)

temp_df = temp_df.withColumn(
    "file_name_only",
    regexp_extract("input_file_name", r"([^/]+$)", 1)
)

display(
    temp_df.select("file_name_only").distinct()
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
display(
    temp_df.select(
        "file_name_only",
        "Week number"
    )
    .distinct()
    .orderBy("Week number")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Show number of weeks by category
from pyspark.sql.functions import countDistinct

temp_df = temp_df.withColumn(
    "Category",
    substring(col("file_name_only"), 5, 4)
)

display(
    temp_df
        .groupBy("Category")
        .agg(countDistinct("Week number").alias("Week_Count"))
)

### END CODE TO Check week numbers in all categories ###

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
### CODE TO CREATE CALENDAR TABLE ###

temp_df = temp_df.select("Category","Week number", "Top Level Field Name", "Response Name")
temp_df = temp_df.filter(col('Top Level Field Name').isin(['Rounds',
                'Quarterly','Roll4','Roll8','Roll12','Roll24','Roll26','Roll52',
                'Disc1','Disc4','Disc8','Disc12']))

temp_df = temp_df.dropDuplicates()

temp_df = temp_df.withColumn(
    "Response Name",
    regexp_replace(col("Response Name"), r"[^\x00-\x7F]", "")
)

display(temp_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
display(temp_df.select("Week number").distinct())

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# QUARTERS

quarter_df = temp_df.filter(col('Top Level Field Name') == "Quarterly")
#quarter_df = quarter_df.dropDuplicates()
quarter_df.count()
#display(quarter_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql.functions import lit, concat, substring, concat_ws

# Rename column
quarter_df = quarter_df.withColumnRenamed("Top Level Field Name", "Period Type")

# Replace "Quarterly" with "Quarter"
quarter_df = quarter_df.withColumn("Period Type",
                        regexp_replace(col("Period Type"), "Quarterly", "Discrete Quarter"))

# Create duplicate column Time Period 
quarter_df = quarter_df.withColumn("Time Period", col("Response Name"))

# Extract year (2023, 2024, etc.)
quarter_df = quarter_df.withColumn("Year", substring(col("Time Period"), 3, 4))

# Extract quarter number (1, 2, etc.)
quarter_df = quarter_df.withColumn("Quarter Number", substring(col("Time Period"), 2, 1))

# Modify Time Period as "Q1 2023"
quarter_df = quarter_df.withColumn("Time Period", concat_ws(" ", 
                                concat(lit("Q"), col("Quarter Number")), col("Year")))

# Add extra time period columns
quarter_df = quarter_df.withColumn("Time Period Multiselect", col("Time Period"))
quarter_df = quarter_df.withColumn("Time Period_2", col("Time Period"))

# Add blank columns Starting Week Number & Ending Week Number (to match schema)
quarter_df = quarter_df.withColumn("Starting Week Number", lit(""))
quarter_df = quarter_df.withColumn("Ending Week Number", lit(""))

# Time Period Sort Order (YYYY + quarter)
quarter_df = quarter_df.withColumn("Time Period Sort Order", concat(col("Year"), 
                                                                col("Quarter Number")))

# Period Type Sort Order constant
quarter_df = quarter_df.withColumn("Period Type Sort Order", lit(2))

display(quarter_df)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# MAT

mat_df = temp_df.filter(col('Top Level Field Name') == "Quarterly")

mat_df.count()
#display(mat_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col, substring
from pyspark.sql.types import IntegerType

# Extract year (2023, 2024, etc.)
mat_df = mat_df.withColumn(
    "Year",
    substring(col("Response Name"), 3, 4).cast(IntegerType())
)

# Extract quarter number (1, 2, etc.)
mat_df = mat_df.withColumn(
    "Quarter Number",
    substring(col("Response Name"), 2, 1).cast(IntegerType())
)

# Window based on Year & Quarter Number
w = Window.partitionBy("Category").orderBy(
    "Year",
    "Quarter Number"
)

# Add Ranks
mat_df = mat_df.withColumn(
    "quarter_rank",
    dense_rank().over(w)
)

display(mat_df)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# self join to create MATs

df1 = mat_df.select("Category", mat_df["Response Name"].alias("Time Period"),
                        "Year", "Quarter Number",
                        mat_df["quarter_rank"].alias("a_quarter_rank"))
df1 = df1.drop_duplicates()

df2 = mat_df.select(mat_df["Category"].alias("Category_b"), "Week number",
                        mat_df["Response Name"].alias("Quarter"),
                        mat_df["quarter_rank"].alias("b_quarter_rank"))

join_condition = ( (col("a_quarter_rank") >= col("b_quarter_rank"))
                 & (col("a_quarter_rank") <= col("b_quarter_rank") + lit(3))
                 & (col("Category") == col("Category_b")) )


mat_df = df1.join(df2, join_condition, how = "inner")

display(mat_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Add Period Type Column
mat_df = mat_df.withColumn("Period Type", lit("MAT"))

# Add blank column Response Name (to match schema)
mat_df = mat_df.withColumn("Response Name", lit(""))

# Modify Time Period Column
mat_df = mat_df.withColumn(
    "Time Period",
    concat(
        lit("MAT (Q"),
        col("Quarter Number"),
        lit(" '"),
        substring(col("Year"), 3, 2),
        lit(")")
    )
)

# Add extra time period columns
mat_df = mat_df.withColumn("Time Period Multiselect", col("Time Period"))
mat_df = mat_df.withColumn("Time Period_2", col("Time Period"))

# Add blank columns Starting Week Number & Ending Week Number (to match schema)
mat_df = mat_df.withColumn("Starting Week Number", lit(""))
mat_df = mat_df.withColumn("Ending Week Number", lit(""))

# Time Period Sort Order (YYYY + Quarter)
mat_df = mat_df.withColumn(
    "Time Period Sort Order",
    concat(col("Year"), col("Quarter Number"))
)

# Period Type Sort Order constant
mat_df = mat_df.withColumn("Period Type Sort Order", lit(1))

mat_df = mat_df.select(
    "Category",
    "Week number",
    "Period Type",
    "Response Name",
    "Time Period",
    "Year",
    "Quarter Number",
    "Time Period Multiselect",
    "Time Period_2",
    "Starting Week Number",
    "Ending Week Number",
    "Time Period Sort Order",
    "Period Type Sort Order"
)

display(mat_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Week End Date

wed_df = temp_df.filter(
    col("Top Level Field Name") == "Disc1"
)

wed_df = wed_df.select(
    "Week number",
    "Response Name"
).dropDuplicates()

# UL Lakehouse Metadata
ul_meta_df = spark.read.csv(
    "Files/Meta Data",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

# Historical Metadata
hist_meta_df = spark.read.csv(
    "Files/META DATA_202301_202630",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

# July Metadata
jul_meta_df = spark.read.csv(
    "Files/META DATA_202627_202630",
    sep=",",
    inferSchema=True,
    header=True,
    encoding="UTF-16"
)

# Combine all Metadata
temp_meta_df = (
    ul_meta_df
        .unionByName(hist_meta_df)
        .unionByName(jul_meta_df)
)

temp_meta_df = temp_meta_df.filter(
    col("Top Level Field Name") == "Disc1"
)

temp_meta_df = (
    temp_meta_df
        .select(
            "Response Name",
            "Response Label"
        )
        .dropDuplicates()
)

temp_meta_df = temp_meta_df.withColumn(
    "Response Name",
    regexp_replace(
        col("Response Name"),
        r"[^\x00-\x7F]",
        ""
    )
)

temp_meta_df = temp_meta_df.withColumn(
    "Response Label",
    regexp_replace(
        col("Response Label"),
        r"[^\x00-\x7F]",
        ""
    )
)

wed_df = wed_df.join(
    temp_meta_df,
    on=["Response Name"],
    how="left"
)

wed_df = wed_df.withColumn(
    "Week End Date",
    concat_ws(
        "-",
        substring(col("Response Label"), 16, 2),  # MM
        substring(col("Response Label"), 13, 2),  # DD
        substring(col("Response Label"), 2, 4)    # YYYY
    )
)

wed_df = wed_df.select(
    "Week number",
    "Week End Date"
)

display(wed_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# ROLLINGS

Roll_df = temp_df.filter(col('Top Level Field Name').isin(['Roll4','Roll8','Roll12','Roll24','Roll26','Roll52']))
Roll_df.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql.functions import lit, concat, substring, concat_ws

# Rename column
Roll_df = Roll_df.withColumnRenamed("Top Level Field Name", "Period Type")

# Replace "Roll" with "Rolling "
Roll_df = Roll_df.withColumn(
    "Period Type",
    regexp_replace(col("Period Type"), "Roll", "Rolling ")
)

# Create column Time Period with Ending Week Number
Roll_df = Roll_df.withColumn(
    "Time Period",
    substring(col("Response Name"), 12, 6)
)

# Joining Week End Date
wed_df = wed_df.withColumnRenamed("Week number", "Time Period")

Roll_df = Roll_df.join(
    wed_df,
    on=["Time Period"],
    how="left"
)

# Add column Year
Roll_df = Roll_df.withColumn(
    "Year",
    substring(col("Response Name"), 12, 4)
)

# Add blank column Quarter Number
Roll_df = Roll_df.withColumn(
    "Quarter Number",
    lit(None).cast("string")
)

# Add extra time period columns
Roll_df = Roll_df.withColumn(
    "Time Period Multiselect",
    col("Week End Date")
)

Roll_df = Roll_df.withColumn(
    "Time Period_2",
    col("Time Period")
)

# Starting Week Number
Roll_df = Roll_df.withColumn(
    "Starting Week Number",
    substring(col("Response Name"), 2, 6)
)

# Ending Week Number
Roll_df = Roll_df.withColumn(
    "Ending Week Number",
    col("Time Period")
)

# Time Period Sort Order
Roll_df = Roll_df.withColumn(
    "Time Period Sort Order",
    col("Time Period")
)

# Update Time Period to Week End Date
Roll_df = Roll_df.withColumn(
    "Time Period",
    col("Week End Date")
)

# Period Type Sort Order
Roll_df = Roll_df.withColumn(
    "Period Type Sort Order",
    regexp_replace(col("Period Type"), "Rolling ", "")
)

# Reordering columns
Roll_df = Roll_df.select(
    "Category",
    "Week number",
    "Period Type",
    "Response Name",
    "Time Period",
    "Year",
    "Quarter Number",
    "Time Period Multiselect",
    "Time Period_2",
    "Starting Week Number",
    "Ending Week Number",
    "Time Period Sort Order",
    "Period Type Sort Order"
)

display(Roll_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# DISCRETES

Disc_df = temp_df.filter(col('Top Level Field Name').isin(['Disc4','Disc8','Disc12']))
Disc_df.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql.functions import lit, concat, substring, concat_ws

# Rename column
Disc_df = Disc_df.withColumnRenamed("Top Level Field Name", "Period Type")

# Replace "Disc" with "Discrete "
Disc_df = Disc_df.withColumn(
    "Period Type",
    regexp_replace(col("Period Type"), "Disc", "Discrete ")
)

# Create column Time Period with Ending Week Number
Disc_df = Disc_df.withColumn(
    "Time Period",
    substring(col("Response Name"), 12, 6)
)

# Joining Week End Date
Disc_df = Disc_df.join(wed_df, on=["Time Period"], how="left")

# Add column Year
Disc_df = Disc_df.withColumn(
    "Year",
    substring(col("Response Name"), 12, 4)
)

# Add blank column Quarter Number (to match schema)
Disc_df = Disc_df.withColumn("Quarter Number", lit(""))

# Add extra time period columns
Disc_df = Disc_df.withColumn("Time Period Multiselect", col("Week End Date"))
Disc_df = Disc_df.withColumn("Time Period_2", col("Time Period"))

# Starting Week Number
Disc_df = Disc_df.withColumn(
    "Starting Week Number",
    substring(col("Response Name"), 2, 6)
)

# Ending Week Number
Disc_df = Disc_df.withColumn(
    "Ending Week Number",
    col("Time Period")
)

# Time Period Sort Order
Disc_df = Disc_df.withColumn(
    "Time Period Sort Order",
    col("Time Period")
)

# Update Time Period to Week End Date
Disc_df = Disc_df.withColumn(
    "Time Period",
    col("Week End Date")
)

# Period Type Sort Order
Disc_df = Disc_df.withColumn(
    "Period Type Sort Order",
    lit(6)
)

# Reordering columns
Disc_df = Disc_df.select(
    "Category",
    "Week number",
    "Period Type",
    "Response Name",
    "Time Period",
    "Year",
    "Quarter Number",
    "Time Period Multiselect",
    "Time Period_2",
    "Starting Week Number",
    "Ending Week Number",
    "Time Period Sort Order",
    "Period Type Sort Order"
)

display(Disc_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# ROUNDS

Round_df = temp_df.filter(col('Top Level Field Name') == 'Rounds')
Round_df.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql.functions import lit, concat, substring, concat_ws, length, col
from pyspark.sql import functions as F

# Rename column
Round_df = Round_df.withColumnRenamed("Top Level Field Name", "Period Type")

# Create column Time Period
Round_df = Round_df.withColumn(
    "Time Period",
    col("Response Name")
)

# Add column Year
Round_df = Round_df.withColumn(
    "Year",
    substring(col("Response Name"), -4, 4)
)

# Add blank column Quarter Number (to match schema)
Round_df = Round_df.withColumn(
    "Quarter Number",
    lit(None).cast("string")
)

# Add extra time period columns
Round_df = Round_df.withColumn(
    "Time Period Multiselect",
    col("Time Period")
)

Round_df = Round_df.withColumn(
    "Time Period_2",
    col("Time Period")
)

# Starting Week Number
Round_df = Round_df.withColumn(
    "Starting Week Number",
    lit(None).cast("string")
)

# Ending Week Number
Round_df = Round_df.withColumn(
    "Ending Week Number",
    lit(None).cast("string")
)

# Time Period Sort Order
Round_df = Round_df.withColumn(
    "Time Period Sort Order",
    concat(
        col("Year"),
        F.lpad(
            F.regexp_extract(
                F.col("Time Period"),
                r"^R(\d+)_",
                1
            ),
            2,
            "0"
        )
    )
)

# Period Type Sort Order constant
Round_df = Round_df.withColumn(
    "Period Type Sort Order",
    lit("9")
)

# Reordering columns
Round_df = Round_df.select(
    "Category",
    "Week number",
    "Period Type",
    "Response Name",
    "Time Period",
    "Year",
    "Quarter Number",
    "Time Period Multiselect",
    "Time Period_2",
    "Starting Week Number",
    "Ending Week Number",
    "Time Period Sort Order",
    "Period Type Sort Order"
)

display(Round_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# COMBINING AND SAVING AS A TABLE

Calendar_df = quarter_df.union(mat_df).union(Roll_df).union(Disc_df).union(Round_df)

Calendar_df = Calendar_df.toDF(*(c.replace(' ', '_') for c in Calendar_df.columns))

Calendar_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("CalendarTable_1")


### END CALENDAR TABLE CREATION ###

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# ============================================================
# FINAL QC CHECKS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col

print("=" * 80)
print("FINAL QC CHECKS")
print("=" * 80)

# ============================================================
# QC 1 : DUPLICATES IN CASEDATA
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col

print("\nQC 1 : DUPLICATES IN CASEDATA")

duplicate_check_df = (
    casedata_df
        .groupBy(
            col("`Respondent.Serial`"),
            col("Week number"),
            col("Top Level Field Name"),
            col("First Level Iterator Name"),
            col("First Level Iterator Label"),
            col("First Level Field Name"),
            col("Response Name"),
            col("Second Level Iterator Label"),
            col("Second Level Iterator Name"),
            col("Second Level Field Name"),
            col("Category"),
            col("Respondent Category ID")
        )
        .count()
        .filter(col("count") > 1)
)

duplicate_count = duplicate_check_df.count()

print("Duplicate Groups Found:", duplicate_count)

if duplicate_count > 0:
    display(
        duplicate_check_df.orderBy(
            col("count").desc()
        )
    )
else:
    print("PASS: No duplicate CaseData groups found.")

# ============================================================
# QC 2 : WEIGHTS AFTER PIVOT
# ============================================================

print("\nQC 2 : WEIGHTS AFTER PIVOT")

print("weights_df count :", weights_df.count())
print("base_df count    :", base_df.count())

print("\nweights_df columns:")
print(weights_df.columns)

# ============================================================
# QC 3 : DUPLICATE VALUES AFTER PIVOT
# Example: '1,1' or '0.044,0.044'
# ============================================================

print("\nQC 3 : DUPLICATE VALUES AFTER PIVOT")

for c in weights_df.columns:

    if c not in ["Respondent Category ID", "Week number"]:

        duplicate_values = (
            weights_df
                .filter(
                    col(c).cast("string").contains(",")
                )
                .count()
        )

        print(f"{c}: {duplicate_values}")

# ============================================================
# QC 4 : AVAILABLE WEIGHT VARIABLES
# ============================================================

print("\nQC 4 : AVAILABLE WEIGHT VARIABLES")

display(
    casedata_df
        .filter(
            col("Top Level Field Name").isin([
                "WEIGHT",
                "Synthetic",
                "Weight_human",
                "human_weights_squared",
                "boost_factor",
                "Rounds"
            ])
        )
        .groupBy("Top Level Field Name")
        .count()
        .orderBy("Top Level Field Name")
)

# ============================================================
# QC 5 : BOOST_FACTOR / ROUNDS
# ============================================================

print("\nQC 5 : BOOST_FACTOR / ROUNDS")

print("bf_df count :", bf_df.count())
print("bf_df columns :", bf_df.columns)

display(bf_df.limit(20))

# ============================================================
# QC 6 : WEIGHTS vs BASE KEY RECONCILIATION
# ============================================================

print("\nQC 6 : WEIGHTS VS BASE")

weights_keys = (
    weights_df
        .select(
            "Respondent Category ID",
            "Week number"
        )
        .distinct()
)

base_keys = (
    base_df
        .select(
            "Respondent_Category_ID",
            "Week_number"
        )
        .distinct()
        .withColumnRenamed(
            "Respondent_Category_ID",
            "Respondent Category ID"
        )
        .withColumnRenamed(
            "Week_number",
            "Week number"
        )
)

weights_key_count = weights_keys.count()
base_key_count = base_keys.count()

print("weights keys :", weights_key_count)
print("base keys    :", base_key_count)
print("difference   :", weights_key_count - base_key_count)

# ============================================================
# QC 7 : RESPONDENTS IN WEIGHTS BUT NOT IN BASE
# ============================================================

print("\nQC 7 : WEIGHTS NOT IN BASE")

weights_not_in_base = (
    weights_keys.join(
        base_keys,
        ["Respondent Category ID", "Week number"],
        "left_anti"
    )
)

base_not_in_weights = (
    base_keys.join(
        weights_keys,
        ["Respondent Category ID", "Week number"],
        "left_anti"
    )
)

print(
    "weights_not_in_base :",
    weights_not_in_base.count()
)

print(
    "base_not_in_weights :",
    base_not_in_weights.count()
)

display(
    weights_not_in_base
        .withColumn(
            "CategoryPart",
            F.split(
                col("Respondent Category ID"),
                "-"
            ).getItem(0)
        )
        .groupBy("CategoryPart")
        .count()
        .orderBy(F.desc("count"))
)

# ============================================================
# QC 8 : BLANK CATEGORY CHECK
# ============================================================

print("\nQC 8 : BLANK CATEGORY CHECK")

blank_category_count = (
    casedata_df
        .filter(
            col("Category").isNull() |
            (col("Category") == "")
        )
        .count()
)

print(
    "Blank Category Records :",
    blank_category_count
)

display(
    casedata_df
        .groupBy("Category")
        .count()
        .orderBy("Category")
)

# ============================================================
# QC COMPLETE
# ============================================================

print("\nQC COMPLETED")
print("=" * 80)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# ============================================================
# QC : REQUIRED VARIABLES COVERAGE
# Check all synthetic & non-synthetic variables
# across weeks and categories
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col

required_variables = [
    "WEIGHT",
    "Synthetic",
    "Weight_human",
    "human_weights_squared",
    "boost_factor",
    "Rounds",
    "Country",
    "Category",
    "QSex",
    "MRKAgeBands",
    "Age",
    "Ethnicity",
    "SOCMED1_CORE",
    "SOCMED2_CORE",
    "BE6Total",
    "BE7Total",
    "BP1",
    "BP6",
    "COM19Flip",
    "Pack1Core",
    "Q7aYes",
    "upos1",
    "upos2",
    "UsageTab",
    "UPOS2_HeardOf",
    "BHCAsked"
]

print("=" * 80)
print("QC : VARIABLE COVERAGE CHECK")
print("=" * 80)

# ============================================================
# Variables available
# ============================================================

available_vars = (
    casedata_df
        .select("Top Level Field Name")
        .distinct()
)

display(
    available_vars.orderBy("Top Level Field Name")
)

# ============================================================
# Missing variables check
# ============================================================

available_var_list = [
    row["Top Level Field Name"]
    for row in available_vars.collect()
]

missing_vars = sorted(
    list(
        set(required_variables) -
        set(available_var_list)
    )
)

print("\nMissing Variables:")
print(missing_vars)

# ============================================================
# Variable coverage by week
# ============================================================

print("\nVARIABLE COVERAGE BY WEEK")

display(
    casedata_df
        .filter(
            col("Top Level Field Name").isin(required_variables)
        )
        .groupBy(
            "Week number",
            "Top Level Field Name"
        )
        .agg(
            F.count("*").alias("Record_Count")
        )
        .orderBy(
            "Week number",
            "Top Level Field Name"
        )
)

# ============================================================
# Variable coverage by Category
# ============================================================

print("\nVARIABLE COVERAGE BY CATEGORY")

display(
    casedata_df
        .filter(
            col("Top Level Field Name").isin(required_variables)
        )
        .groupBy(
            "Category",
            "Top Level Field Name"
        )
        .agg(
            F.count("*").alias("Record_Count")
        )
        .orderBy(
            "Category",
            "Top Level Field Name"
        )
)

# ============================================================
# Synthetic Variables Only
# ============================================================

print("\nSYNTHETIC VARIABLES COVERAGE")

display(
    casedata_df
        .filter(
            col("Top Level Field Name").isin([
                "Synthetic",
                "Weight_human",
                "human_weights_squared",
                "boost_factor",
                "Rounds"
            ])
        )
        .groupBy(
            "Category",
            "Week number",
            "Top Level Field Name"
        )
        .agg(
            F.count("*").alias("Record_Count")
        )
        .orderBy(
            "Category",
            "Week number",
            "Top Level Field Name"
        )
)

print("\nQC COMPLETED")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
tables = [
    "basetable_1",
    "facttable_1",
    "questiontable_1",
    "ethnicitytable_1",
    "dptdailytable_1",
    "dptregularlytable_1",
    "calendartable_1"
]

for t in tables:
    
    df = spark.read.table(f"dbo.{t}")

    print(
        f"{t}: {df.count():,}"
    )

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F

base_df = spark.read.table("dbo.basetable_1")

display(
    base_df
        .groupBy("Category")
        .agg(
            F.countDistinct(
                "Respondent_Category_ID"
            ).alias("Respondents"),
            F.countDistinct(
                "Week_number"
            ).alias("Weeks")
        )
        .orderBy("Category")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
display(
    base_df
        .groupBy(
            "Respondent_Category_ID",
            "Week_number"
        )
        .count()
        .filter("count > 1")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
fact_df = spark.read.table("dbo.facttable_1")

display(
    fact_df
        .groupBy("TopLevelField")
        .count()
        .orderBy("TopLevelField")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
display(
    base_df
        .select("Week_number")
        .distinct()
        .orderBy("Week_number")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql.functions import col

base_df = spark.read.table("dbo.basetable_1")

display(
    base_df
        .filter(col("Category") == "FMON")
        .select("Week_number")
        .distinct()
        .orderBy("Week_number")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F

fact_df = spark.read.table("dbo.facttable_1")

display(
    fact_df
        .groupBy("TopLevelField")
        .agg(
            F.count("*").alias("Records")
        )
        .orderBy("TopLevelField")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F

fact_df = spark.read.table("dbo.facttable_1")

display(
    fact_df
        .groupBy("Synthetic")
        .agg(
            F.count("*").alias("Records")
        )
)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

fact_df = spark.read.table("dbo.facttable_1")

display(
    fact_df
        .filter(
            col("Synthetic").isNull()
        )
        .groupBy(
            "WeekNumber",
            "TopLevelField"
        )
        .agg(
            F.count("*").alias("Records")
        )
        .orderBy(
            "WeekNumber",
            "TopLevelField"
        )
)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
display(
    fact_df
        .filter(
            col("Synthetic").isNull()
        )
        .groupBy("WeekNumber")
        .agg(
            F.count("*").alias("Records")
        )
       .orderBy("WeekNumber")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql.functions import col

fact_df = spark.read.table("dbo.facttable_1")

display(
    fact_df
        .filter(col("Synthetic").isNull())
        .select(
            "Respondent_Category_ID",
            "WeekNumber",
            "WEIGHT",
            "Weight_human",
            "human_weights_squared"
        )
        .limit(100)
)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

fact_df = spark.read.table("dbo.facttable_1")

display(
    fact_df
        .filter(col("Synthetic").isNull())
        .withColumn(
            "Category",
            F.split(
                col("Respondent_Category_ID"),
                "-"
            ).getItem(0)
        )
        .groupBy("Category")
        .agg(
            F.countDistinct(
                "Respondent_Category_ID"
            ).alias("Respondents"),
            F.count("*").alias("Records")
        )
        .orderBy("Category")
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
base_df = spark.read.table("dbo.basetable_1")

display(
    base_df
        .select("Week_number")
        .distinct()
        .orderBy("Week_number")
)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import functions as F

base_df = spark.read.table("dbo.basetable_1")

display(
    base_df
        .filter(
            F.col("Week_number") >= 202631
        )
        .groupBy(
            "Week_number",
            "Category"
        )
        .agg(
            F.countDistinct(
                "Respondent_Category_ID"
            ).alias("Respondents")
        )
        .orderBy(
            "Week_number",
            "Category"
        )
)

StatementMeta(, , -1, Waiting, , Waiting, True)